# Secant Method
> **Numerical Methods for Engineering** | Module 01 - Root Finding | `03_Secant.ipynb`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bpatinoa/numerical-methods-for-engineering/blob/main/01-Root-Finding/03_Secant.ipynb)

---

## Learning Objectives

After completing this notebook you will be able to:
- Derive the Secant update formula as a **finite-difference Newton-Raphson**.
- Prove that the convergence order is the **golden ratio** $\varphi \approx 1.618$.
- Implement the algorithm requiring **no explicit derivative**.
- Apply the method to problems in general mathematics, chemistry, and telecommunications.
- Compare Secant, Newton-Raphson, and Bisection quantitatively.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize':(11,5),'font.size':12,
    'axes.grid':True,'grid.alpha':0.35,'lines.linewidth':2})
print('Libraries loaded OK')

---
## 1. Theoretical Background

### 1.1 Motivation: Newton-Raphson Without the Derivative

The Newton-Raphson update is:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

When $f'$ is unavailable, expensive, or hard to derive analytically, we replace it with a
**finite-difference approximation** using the two most recent iterates:

$$f'(x_n) \approx \frac{f(x_n) - f(x_{n-1})}{x_n - x_{n-1}}$$

Substituting gives the **Secant Method update**:

$$\boxed{x_{n+1} = x_n - f(x_n)\,\frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}}$$

**Geometric interpretation:** $x_{n+1}$ is the $x$-intercept of the **secant line** through
the two points $(x_{n-1},\,f(x_{n-1}))$ and $(x_n,\,f(x_n))$.

---

### 1.2 Convergence Analysis — The Golden Ratio Order

**Theorem:** Near a simple root $r$, the Secant method converges with order

$$p = \varphi = \frac{1+\sqrt{5}}{2} \approx 1.618 \quad \text{(golden ratio)}$$

**Proof sketch:** The error satisfies the recurrence:

$$e_{n+1} \approx -\frac{f''(r)}{2f'(r)}\,e_n\,e_{n-1}$$

Assume convergence of order $p$, so $|e_{n+1}| \sim C|e_n|^p$. Then:

$$C|e_{n-1}|^{p^2} \sim C\cdot C\,|e_{n-1}|^{p+1}$$

Matching exponents: $p^2 = p + 1 \implies p^2 - p - 1 = 0 \implies p = \dfrac{1+\sqrt{5}}{2} \approx 1.618$ $\square$

> **Comparison:** Bisection $p=1$, Secant $p\approx1.618$, Newton-Raphson $p=2$.
> Secant offers a superlinear speed-up over Bisection at the cost of just one extra function evaluation per step.

---

### 1.3 Cost Comparison Per Digit of Accuracy

| Method | Order $p$ | Evaluations/iter | $f'$ needed? |
|--------|-----------|-----------------|--------------|
| Bisection | 1 | 1 | No |
| Secant | 1.618 | 1 | No |
| Newton-Raphson | 2 | 1 + 1 for $f'$ | **Yes** |

Each Secant iteration costs only **one** new function evaluation (the other point is reused).

---

### 1.4 Stopping Criteria

| Criterion | Formula |
|-----------|---------|
| Relative step | $\|x_{n+1}-x_n\|/\max(\|x_{n+1}\|,1) < \varepsilon$ |
| Function value | $\|f(x_{n+1})\| < \delta$ |

---

### 1.5 Pseudocode

```
INPUT:  f, x0, x1, tol, max_iter

FOR i = 1 TO max_iter:
    IF |f(x1) - f(x0)| < tiny:
        STOP  -- nearly parallel secant line
    x2  <-  x1 - f(x1) * (x1 - x0) / (f(x1) - f(x0))
    IF |x2 - x1| / max(|x2|, 1) < tol:
        RETURN x2
    x0, x1 <- x1, x2
END FOR
RETURN x2
```

---

### 1.6 Advantages and Limitations

| Advantages | Limitations |
|------------|-------------|
| Superlinear convergence ($p\approx1.618$) | No guaranteed convergence (no bracket maintained) |
| **No derivative required** | Needs two starting points $x_0, x_1$ |
| One function evaluation per iteration | May diverge if starting points are poorly chosen |
| Naturally generalises to Broyden for systems | Slower than Newton-Raphson for same number of iterations |


In [ ]:
def secant(f, x0, x1, tol=1e-8, max_iter=50, verbose=False):
    # Secant Method: find root of f using two starting points x0, x1.
    hist = {'iterates':[x0,x1], 'errors':[], 'f_values':[f(x0),f(x1)]}
    if verbose:
        print(f"{'n':>4}  {'x_n':>18}  {'f(x_n)':>14}  {'|step|':>12}")
        print('-'*56)
    for i in range(1, max_iter+1):
        f0, f1 = f(x0), f(x1)
        denom = f1 - f0
        if abs(denom) < 1e-14:
            raise ZeroDivisionError('f(x1)-f(x0)~0 -- secant line is horizontal.')
        x2   = x1 - f1*(x1 - x0)/denom
        step = abs(x2 - x1)
        rel  = step / max(abs(x2), 1.0)
        hist['iterates'].append(x2)
        hist['errors'].append(step)
        hist['f_values'].append(f(x2))
        if verbose:
            print(f'{i:>4}  {x2:>18.12f}  {f(x2):>14.4e}  {step:>12.4e}')
        if rel < tol:
            break
        x0, x1 = x1, x2
    return x2, hist


def estimate_order(errors, true_root, iterates):
    # Empirically estimate convergence order from true errors.
    true_errs = [abs(x - true_root) for x in iterates if abs(x-true_root) > 1e-16]
    orders = []
    for k in range(1, len(true_errs)-1):
        if true_errs[k] > 0 and true_errs[k-1] > 0 and true_errs[k+1] > 0:
            p = np.log(true_errs[k+1]/true_errs[k]) / np.log(true_errs[k]/true_errs[k-1])
            orders.append(p)
    return orders

---
## 2. Examples

### 2.1 General Mathematical Example

Find the root of $f(x) = x^3 - x - 2$ (same as Bisection notebook for direct comparison).

**Starting points:** $x_0 = 1.0$, $x_1 = 2.0$ (endpoints of the bracket used in Bisection).

**Exact root:** $x^* \approx 1.52137970680\ldots$


In [ ]:
f1 = lambda x: x**3 - x - 2
x_star = 1.5213797068045675

root1, h1 = secant(f1, x0=1.0, x1=2.0, tol=1e-12, verbose=True)
print(f'\nRoot      : {root1:.14f}')
print(f'Exact     : {x_star:.14f}')
print(f'Error     : {abs(root1 - x_star):.2e}')
print(f'Iterations: {len(h1["errors"])}')

# Empirical order estimate
orders1 = estimate_order(h1['errors'], x_star, h1['iterates'])
if orders1:
    print(f'Estimated convergence order (last 3 iters): {np.mean(orders1[-3:]):.4f}  (golden ratio = {(1+5**0.5)/2:.4f})')

# Plot: secant lines
x_range = np.linspace(0.5, 2.5, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_range, f1(x_range), color='steelblue', label=r'$f(x)=x^3-x-2$')
ax.axhline(0, color='k', lw=0.8, ls='--')
iters = h1['iterates']
colors_s = plt.cm.cool(np.linspace(0, 1, min(5, len(iters)-1)))
for k in range(min(4, len(iters)-2)):
    x0k, x1k = iters[k], iters[k+1]
    slope = (f1(x1k)-f1(x0k))/(x1k-x0k) if abs(x1k-x0k)>1e-10 else 0
    sec_y = f1(x0k) + slope*(x_range - x0k)
    ax.plot(x_range, sec_y, color=colors_s[k], lw=1.2, ls='--', alpha=0.7,
            label=f'Secant iter {k+1}')
ax.scatter(iters[:-1], [f1(x) for x in iters[:-1]], color='red', s=40, zorder=5)
ax.axvline(root1, color='green', ls=':', lw=2, label=f'Root={root1:.5f}')
ax.set(xlim=(0.8,2.2), ylim=(-3,5), xlabel='x', ylabel='f(x)',
       title='Secant Line Iterations'); ax.legend(fontsize=9)

ax = axes[1]
true_errs = [abs(x-x_star) for x in h1['iterates'][2:]]
ax.semilogy(range(1, len(true_errs)+1), true_errs, 'o-', color='steelblue', ms=5, label='True error')
ax.set(xlabel='Iteration', ylabel='|x_n - r| (log)', title='Superlinear Convergence')
ax.legend()

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/secant_math.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.2 Chemistry Application - Vapour Pressure and Normal Boiling Point

**Background:** The **Antoine equation** relates vapour pressure $P^*$ to temperature $T$:

$$\log_{10}(P^*) = A - \frac{B}{C + T}$$

where $T$ is in **°C** and $P^*$ in **mmHg** (Antoine constants $A, B, C$ are substance-specific).

Finding the **normal boiling point** (where $P^* = 760$ mmHg, i.e., 1 atm) requires solving:

$$\boxed{f(T) = A - \frac{B}{C + T} - \log_{10}(760) = 0}$$

The derivative $f'(T) = B/(C+T)^2$ is computable but messy — the Secant method avoids it entirely.

**Case study: Ethanol (C2H5OH)**

| Constant | Value |
|----------|-------|
| $A$ | 8.20417 |
| $B$ | 1642.89 |
| $C$ | 230.300 |

Expected boiling point: $T^* \approx 78.37\,^\circ\text{C}$.
Two starting guesses: $T_0 = 60\,^\circ\text{C}$, $T_1 = 95\,^\circ\text{C}$.


In [ ]:
# Antoine constants for ethanol (Perry's Chemical Engineers' Handbook)
A_eth, B_eth, C_eth = 8.20417, 1642.89, 230.300
P_target = 760.0   # mmHg (1 atm)

f_antoine = lambda T: A_eth - B_eth/(C_eth + T) - np.log10(P_target)

# Analytical boiling point (solve directly)
T_boil_exact = B_eth / (A_eth - np.log10(P_target)) - C_eth
print(f'Analytical boiling point: {T_boil_exact:.4f} degC')

root2, h2 = secant(f_antoine, x0=60.0, x1=95.0, tol=1e-12, verbose=True)
print(f'\nSecant result : {root2:.8f} degC')
print(f'Exact         : {T_boil_exact:.8f} degC')
print(f'Error         : {abs(root2 - T_boil_exact):.2e} degC')
print(f'Iterations    : {len(h2["errors"])}')

# Plot: P* vs T and convergence
T_range = np.linspace(50, 110, 400)
P_range = 10**(A_eth - B_eth/(C_eth + T_range))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(T_range, P_range, color='teal', label='Vapour pressure $P^*(T)$')
ax.axhline(P_target, color='crimson', ls='--', label='P* = 760 mmHg (1 atm)')
ax.axvline(root2, color='darkorange', ls=':', lw=2,
           label=f'$T_{{bp}}$ = {root2:.2f} °C')
ax.scatter([root2], [P_target], color='darkorange', s=80, zorder=5)
ax.set(xlabel='Temperature (°C)', ylabel='Vapour Pressure (mmHg)',
       title='Ethanol Boiling Point via Antoine Equation')
ax.legend()

ax = axes[1]
ax.semilogy(range(1, len(h2['errors'])+1), h2['errors'],
            'o-', color='teal', ms=5, label='|step|')
ax.set(xlabel='Iteration', ylabel='|step| (log)',
       title='Convergence - Antoine Equation')
ax.legend()

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/secant_chem.png', dpi=120, bbox_inches='tight')
plt.show()

### 2.3 Telecommunications Application - Coaxial Cable Frequency-Dependent Attenuation

**Background:** The total attenuation of a coaxial cable (RG-6/U type) has two components:

$$\alpha(f) = \underbrace{\alpha_c\,\sqrt{f}}_{\text{conductor loss}} + \underbrace{\alpha_d\,f}_{\text{dielectric loss}} \quad \text{(dB / 100 m)}$$

where $f$ is frequency in **MHz**. Given target attenuation $\alpha^* = 30$ dB/100 m, find:

$$\boxed{g(f) = \alpha_c\sqrt{f} + \alpha_d\,f - \alpha^* = 0}$$

The derivative $g'(f) = \alpha_c/(2\sqrt{f}) + \alpha_d$ exists but the Secant method avoids it
— useful when cable parameters are measured empirically and not given analytically.

| Parameter | Value | Description |
|-----------|-------|-------------|
| $\alpha_c$ | 0.996 | Conductor loss coefficient (dB·m$^{-0.5}$·100m$^{-1}$) |
| $\alpha_d$ | $1.91\times10^{-4}$ | Dielectric loss coefficient (dB·MHz$^{-1}$·100m$^{-1}$) |
| $\alpha^*$ | 30 dB/100 m | Target attenuation |

Starting guesses: $f_0 = 500$ MHz, $f_1 = 1500$ MHz.


In [ ]:
alpha_c  = 0.996       # dB / (MHz^0.5 * 100m)
alpha_d  = 1.91e-4     # dB / (MHz * 100m)
alpha_target = 30.0    # dB / 100m

def attenuation(f_MHz):
    return alpha_c * np.sqrt(f_MHz) + alpha_d * f_MHz

g_coax = lambda f: attenuation(f) - alpha_target

root3, h3 = secant(g_coax, x0=500.0, x1=1500.0, tol=1e-8, verbose=True)
print(f'\nFrequency where alpha = {alpha_target} dB/100m : {root3:.2f} MHz')
print(f'Check: alpha({root3:.1f} MHz) = {attenuation(root3):.6f} dB/100m')
print(f'Iterations: {len(h3["errors"])}')

f_vals = np.linspace(100, 2000, 800)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(f_vals, attenuation(f_vals), color='royalblue', label=r'$\alpha(f)$')
ax.axhline(alpha_target, color='crimson', ls='--', label=f'Target = {alpha_target} dB/100m')
ax.axvline(root3, color='darkorange', ls=':', lw=2, label=f'f* = {root3:.1f} MHz')
ax.scatter([root3], [alpha_target], color='darkorange', s=80, zorder=5)
ax.set(xlabel='Frequency (MHz)', ylabel='Attenuation (dB/100 m)',
       title='RG-6/U Coaxial Cable Attenuation')
ax.legend()

ax = axes[1]
ax.semilogy(range(1, len(h3['errors'])+1), h3['errors'],
            'o-', color='royalblue', ms=5, label='|step|')
ax.set(xlabel='Iteration', ylabel='|step| (log)', title='Convergence - Telecom')
ax.legend()

plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/secant_telecom.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Three-Way Comparison: Secant vs. Newton-Raphson vs. Bisection

All three methods applied to $f(x) = \cos x - x$ (true root $r \approx 0.73909$).
Bisection: bracket $[0,1]$; Newton-Raphson: $x_0 = 0.5$, $f'(x)=-\sin x-1$; Secant: $x_0=0$, $x_1=1$.


In [ ]:
f_cmp  = lambda x: np.cos(x) - x
df_cmp = lambda x: -np.sin(x) - 1
r_cmp  = 0.7390851332151607

# Bisection (local copy)
def bisection_cmp(f, a, b, tol=1e-15):
    errs=[]
    for _ in range(60):
        c=(a+b)/2; e=(b-a)/2; errs.append(e)
        if e<tol or f(c)==0: break
        if f(a)*f(c)<0: b=c
        else: a=c
    return c, errs

# Newton-Raphson (local copy)
def nr_cmp(f, df, x0, tol=1e-15):
    x=x0; errs=[]
    for _ in range(20):
        xn=x-f(x)/df(x); errs.append(abs(xn-x))
        if abs(xn-x)<tol: break
        x=xn
    return xn, errs

_, e_bis = bisection_cmp(f_cmp, 0.0, 1.0)
_, e_nr  = nr_cmp(f_cmp, df_cmp, 0.5)
_, h_sec = secant(f_cmp, x0=0.0, x1=1.0, tol=1e-15)
e_sec = h_sec['errors']

print(f'Bisection iterations     : {len(e_bis)}')
print(f'Newton-Raphson iterations: {len(e_nr)}')
print(f'Secant iterations        : {len(e_sec)}')
print(f'NR speed-up vs Bisection : {len(e_bis)/len(e_nr):.1f}x')
print(f'Secant speed-up vs Bisect: {len(e_bis)/len(e_sec):.1f}x')

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(range(1, len(e_bis)+1), e_bis,  'o--', color='steelblue', ms=4, label=f'Bisection ({len(e_bis)} iters)')
ax.semilogy(range(1, len(e_sec)+1), e_sec,  's-',  color='darkorange', ms=5, label=f'Secant ({len(e_sec)} iters)')
ax.semilogy(range(1, len(e_nr)+1),  e_nr,   '^-',  color='crimson',    ms=6, label=f'Newton-Raphson ({len(e_nr)} iters)')
ax.set(xlabel='Iteration', ylabel='Step size (log)',
       title=r'Bisection vs. Secant vs. Newton-Raphson  --  $f(x)=\cos x - x$')
ax.legend()
plt.tight_layout()
plt.savefig('/sessions/epic-eloquent-mccarthy/mnt/outputs/secant_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Student Exercises

---

**Exercise 1 - Transcendental equation**
Apply the Secant method to $g(x) = x - \tan x$ in $(3\pi/2,\;5\pi/2)$.
Use $x_0 = 5.0$, $x_1 = 5.5$. Compare the number of iterations with Newton-Raphson ($f'=1-\sec^2 x$).

---

**Exercise 2 - Clausius-Clapeyron (Chemistry)**
The vapour pressure of benzene (C6H6) follows the Antoine equation with
$A=6.89272$, $B=1203.531$, $C=219.888$.
a) Find the temperature at which $P^*=400$ mmHg using the Secant method.
b) Estimate the enthalpy of vaporisation using the Clausius-Clapeyron equation between the two found temperatures.

---

**Exercise 3 - Chebyshev filter bandwidth (Telecom)**
A 3rd-order Chebyshev lowpass filter with 1 dB ripple has transfer function magnitude:

$$|H(j\omega)|^2 = \frac{1}{1 + \varepsilon^2\,T_3^2(\omega/\omega_p)}$$

where $T_3(x) = 4x^3 - 3x$ and $\varepsilon^2 = 10^{0.1} - 1$.
Find the $-3$ dB frequency $\omega_{3dB}/\omega_p$ (where $|H|^2 = 0.5$) using the Secant method.

---

**Exercise 4 - Convergence order verification**
For the general example ($f(x)=x^3-x-2$, $x_0=1$, $x_1=2$):
a) Compute the empirical convergence order using consecutive true errors:
   $p_k \approx \log|e_{k+1}|/\log|e_k|$
b) Verify that $p \to \varphi = (1+\sqrt{5})/2$.
c) How does this compare to the rate achieved by Newton-Raphson on the same problem?

---

**Exercise 5 - Broyden's Method (extension)**
The Secant method for *systems* $\mathbf{F}(\mathbf{x})=\mathbf{0}$ is Broyden's method.
Research and implement Broyden's Good Method for the 2x2 system:
$f_1(x,y) = x^2 + y - 11 = 0$ and $f_2(x,y) = x + y^2 - 7 = 0$.
Compare with Newton's method for systems.


---
## 5. References

1. **Chapra, S. C., & Canale, R. P.** (2015). *Numerical Methods for Engineers* (7th ed., pp. 135-140).
   McGraw-Hill Education.
   *Section 5.3.2 introduces the Secant method as a modification of Newton-Raphson; §5.3.3 presents
   the modified secant (finite-difference N-R) used in Exercise 3.*

2. **Burden, R. L., Faires, J. D., & Burden, A. M.** (2016). *Numerical Analysis* (10th ed., pp. 80-87).
   Cengage Learning.
   *Theorem 2.8 provides the rigorous proof that the convergence order is the golden ratio $\varphi$,
   following the derivation in Section 1.2 of this notebook.*

3. **Ralston, A., & Rabinowitz, P.** (2001). *A First Course in Numerical Analysis* (2nd ed., pp. 338-345).
   Dover Publications.
   *Section 8.3 analyses the Secant method via divided differences, connecting it to the general
   theory of multi-point iterative methods. Contains the error constant derivation.*

4. **Reid, R. C., Prausnitz, J. M., & Poling, B. E.** (1987). *The Properties of Gases and Liquids*
   (4th ed., Appendix A). McGraw-Hill.
   *Primary source for the Antoine equation constants used in Example 2.2. Table A-2 lists
   $A$, $B$, $C$ for over 200 substances including ethanol.*

5. **Pozar, D. M.** (2012). *Microwave Engineering* (4th ed., pp. 49-58). John Wiley & Sons.
   *Section 2.1 derives the frequency-dependent conductor and dielectric losses in coaxial lines,
   the physical basis of the $\alpha_c\sqrt{f}+\alpha_d f$ model used in Example 2.3.*
